# 04_Metrics.ipynb: Prospective Replay & Rigorous Dual-Branch Evaluation (Version 2)
Tuân thủ nghiêm ngặt AGENTS.md và PROCESS.md.
Đánh giá độc lập 2 nhánh:
- Nhánh A: Dự báo rủi ro sự kiện QC trong cửa sổ tương lai [origin + L, origin + L + H)
- Nhánh B: Sàng lọc rủi ro trực tuyến (QC Triage) cho mẫu hiện tại trước khi có kết quả QC

Lưu ý: 314 quan sát cuối cùng đã từng được quan sát trong tiền sử nghiên cứu trước đây (retrospective evaluation, not freshly blind). Mọi siêu tham số và mô hình đã được đóng băng từ 03_Train.

In [1]:
# Cell 1: Bootstrap & Dynamic Execution of Helper Functions from 03_Train
import os
import sys
import json
import ast
import hashlib
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    roc_auc_score, average_precision_score, recall_score,
    precision_score, f1_score, accuracy_score, balanced_accuracy_score,
    confusion_matrix, mean_absolute_error, mean_squared_error
)

# 1. Bootstrap PROJECT_ROOT and VERSION_DIR
def find_project_root(start_path=None):
    p = Path(start_path).resolve() if start_path else Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if (parent / "AGENTS.md").exists() and (parent / "data" / "secom").exists():
            return parent
    raise RuntimeError(f"Could not locate project root from {p}")

PROJECT_ROOT = find_project_root()
VERSION_DIR = PROJECT_ROOT / "Version 2"
DATA_DIR = PROJECT_ROOT / "data" / "secom"

print(f"[BOOTSTRAP] PROJECT_ROOT: {PROJECT_ROOT}")
print(f"[BOOTSTRAP] VERSION_DIR:  {VERSION_DIR}")

# 2. Runtime Dynamic Execution of tagged helpers from 03_Train.ipynb
train_nb_path = VERSION_DIR / "03_Train.ipynb"
assert train_nb_path.exists(), f"Missing 03_Train notebook at {train_nb_path}"

with open(train_nb_path, "r", encoding="utf-8") as f:
    nb_train = json.load(f)

helper_code_blocks = []
for cell in nb_train.get("cells", []):
    if cell.get("cell_type") == "code":
        tags = cell.get("metadata", {}).get("tags", [])
        if "v2_training_helpers" in tags:
            src = "".join(cell.get("source", []))
            helper_code_blocks.append(src)

assert len(helper_code_blocks) > 0, "No code cells tagged with 'v2_training_helpers' found in 03_Train.ipynb!"
print(f"[BOOTSTRAP] Found {len(helper_code_blocks)} code cells tagged 'v2_training_helpers'. Executing into namespace...")

combined_helpers_src = "\n\n".join(helper_code_blocks)
exec(combined_helpers_src, globals())

# Verify core APIs are now present
required_apis = [
    "load_raw", "make_features", "build_targets", "forecast_stream",
    "apply_forecast_policy", "qc_stream", "apply_qc_policy",
    "interval_metrics", "row_metrics", "incidence", "capacity_threshold", "qc_threshold"
]
for api_name in required_apis:
    assert api_name in globals(), f"Expected helper API '{api_name}' not defined after executing 03 helpers!"

print("[BOOTSTRAP] All core APIs successfully loaded into runtime.")


[BOOTSTRAP] PROJECT_ROOT: D:\AI_Bussiness_Huflit\Industrial_AI_on_UCISECOM
[BOOTSTRAP] VERSION_DIR:  D:\AI_Bussiness_Huflit\Industrial_AI_on_UCISECOM\Version 2
[BOOTSTRAP] Found 3 code cells tagged 'v2_training_helpers'. Executing into namespace...
[BOOTSTRAP] PROJECT_ROOT: D:\AI_Bussiness_Huflit\Industrial_AI_on_UCISECOM
[BOOTSTRAP] VERSION_DIR:  D:\AI_Bussiness_Huflit\Industrial_AI_on_UCISECOM\Version 2
[BOOTSTRAP] PROJECT_ROOT: D:\AI_Bussiness_Huflit\Industrial_AI_on_UCISECOM
[BOOTSTRAP] VERSION_DIR:  D:\AI_Bussiness_Huflit\Industrial_AI_on_UCISECOM\Version 2
[BOOTSTRAP] Successfully imported 5 definition code cells from 02_Preprocessing.ipynb
[BOOTSTRAP] Available functions: load_raw, make_features, build_targets, training_maturity_gate, interval_metrics, row_metrics, capacity_threshold, qc_threshold
[BOOTSTRAP] All core APIs successfully loaded into runtime.


In [2]:
# Cell 2: Step 1 (Artifact Verification & Code SHA), Dev Replay Protocol Audit, & Step 2 (Dev Delay Robustness)
# 1. Load experiment artifact
art_path = VERSION_DIR / "experiment.joblib"
assert art_path.exists(), f"V2 experiment artifact not found at {art_path}"
exp = joblib.load(art_path)
print(f"[STEP 1] Loaded V2 artifact keys: {list(exp.keys())}")

# Check SHA256 of raw data against artifact
raw_df = load_raw(DATA_DIR)
with open(DATA_DIR / "secom.data", "rb") as f:
    act_data_sha = hashlib.sha256(f.read()).hexdigest()
with open(DATA_DIR / "secom_labels.data", "rb") as f:
    act_lbl_sha = hashlib.sha256(f.read()).hexdigest()

assert act_data_sha == exp["raw_hashes"]["secom_data_sha256"], "secom.data SHA256 mismatch with artifact!"
assert act_lbl_sha == exp["raw_hashes"]["secom_labels_sha256"], "secom_labels.data SHA256 mismatch with artifact!"
print(f"[STEP 1] Raw data SHA256 matches artifact perfectly.")

# Capture code-source-only SHA for 02 and 03
p02 = VERSION_DIR / "02_Preprocessing.ipynb"
with open(p02, "r", encoding="utf-8") as f:
    nb02 = json.load(f)
src02 = "".join("".join(c.get("source", [])) for c in nb02.get("cells", []) if c.get("cell_type") == "code")
sha_02 = hashlib.sha256(src02.encode("utf-8")).hexdigest()
sha_03_helpers = hashlib.sha256(combined_helpers_src.encode("utf-8")).hexdigest()

code_source_hashes = {
    "02_preprocessing_code_sha256": sha_02,
    "03_training_helpers_code_sha256": sha_03_helpers,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "sklearn_version": getattr(sys.modules.get("sklearn"), "__version__", "unknown"),
    "prior_holdout_known_exposure": True,
    "client_date": "2026-10-26 Asia/Saigon"
}
print(f"[STEP 1] Environment & Source Code Hashes: {code_source_hashes}")

# Load stream_configs directly from 03 schema
if "stream_configs" in exp:
    cfg_A = dict(exp["stream_configs"]["forecast"])
    cfg_B = dict(exp["stream_configs"]["qc"])
elif "config_A" in exp and "config_B" in exp:
    cfg_A = dict(exp["config_A"])
    cfg_B = dict(exp["config_B"])
else:
    cfg_A = {"delay_h": 6, "rolling_days": 56, "refit_cadence_h": 24, "min_train_origins": 40}
    cfg_B = {"delay_h": 6, "rolling_days": 56, "refit_cadence_h": 24, "min_train_rows": 100}

# Validate exact frozen parameters
assert cfg_A["delay_h"] == 6 and cfg_A["rolling_days"] == 56 and cfg_A["refit_cadence_h"] == 24, f"Invalid cfg_A frozen parameters: {cfg_A}"
assert cfg_B["delay_h"] == 6 and cfg_B["rolling_days"] == 56 and cfg_B["refit_cadence_h"] == 24, f"Invalid cfg_B frozen parameters: {cfg_B}"

wA = exp["selection"]["winner_A"]
wB = exp["selection"]["winner_B"]
sel_H = int(wA["horizon"]) if "horizon" in wA else int(wA.get("horizon_h", 24))
sel_cand_A = wA["candidate"]
sel_budget_A = float(wA["budget"])
sel_lookback_A = int(wA["lookback"]) if "lookback" in wA else int(wA.get("lookback_days", 7))

print(f"[STEP 1] Selected DEV Winner A: Candidate={sel_cand_A}, H={sel_H}h, Budget={sel_budget_A:.2f}, Lookback={sel_lookback_A}d")
print(f"  Coverage: {float(wA['coverage']):.4f} ({int(wA['covered_fail'])}/{int(wA['total_fail'])}), Workload: {float(wA['workload']):.4f} ({int(wA['covered_n'])}/{int(wA['total_n'])})")

sel_cand_B = wB["candidate"]
sel_budget_B = float(wB["budget"])
sel_threshold_budget_B = float(wB.get("threshold_budget", sel_budget_B))
sel_lookback_B = int(wB["lookback"]) if "lookback" in wB else int(wB.get("lookback_days", 14))

print(f"[STEP 1] Selected DEV Winner B: Candidate={sel_cand_B}, QuotaBudget={sel_budget_B:.2f}, ThresholdBudget={sel_threshold_budget_B:.2f}, Lookback={sel_lookback_B}d")
print(f"  Raw Recall: {float(wB['coverage']):.4f} ({int(wB['covered_fail'])}/{int(wB['total_fail'])}), Workload: {float(wB['workload']):.4f} ({int(wB['covered_n'])}/{int(wB['total_n'])})")

# DEV REPLAY PROTOCOL AUDIT FROM SAVED DEV OOS SCORES (NO REFIT, NO RETUNING)
dev_cutoff = pd.Timestamp(exp["protocol"]["dev_cutoff"])
dev_raw = raw_df[raw_df["timestamp"] < dev_cutoff].copy()
dev_eval_spans = [(pd.Timestamp(s), pd.Timestamp(e)) for s, e in exp["temporal_bounds"]["eval_spans"]]
dev_eval_start = pd.Timestamp(exp["temporal_bounds"]["global_eval_start"])
dev_eval_end = pd.Timestamp(exp["temporal_bounds"]["global_eval_end"])

# Branch A Dev Replay
saved_dev_scores_A = exp["selected_dev_scores"]["branch_A"].copy()
dev_tgt_A = build_targets(dev_raw, origins=saved_dev_scores_A.index, horizon_h=sel_H, lead_h=6, delay_h=6, q=0.1, min_event_samples=5)
dev_tgt_A = dev_tgt_A.set_index("origin", drop=False)

w_df_dev_A, summ_dev_A, fold_dev_A = apply_forecast_policy(
    saved_dev_scores_A, dev_raw, dev_tgt_A,
    budget=sel_budget_A, lookback_days=sel_lookback_A,
    eval_span=(dev_eval_start, dev_eval_end),
    disjoint_spans=dev_eval_spans
)

# Branch B Dev Replay
saved_dev_scores_B = exp["selected_dev_scores"]["branch_B"].copy()
w_df_dev_B, summ_dev_B, fold_dev_B = apply_qc_policy(
    saved_dev_scores_B, dev_raw,
    budget=sel_budget_B, lookback_days=sel_lookback_B, threshold_budget=sel_threshold_budget_B,
    report_spans=dev_eval_spans
)

# Assertions matching winner_A & winner_B counts and metrics
assert summ_dev_A["total_n"] == 879 and summ_dev_A["total_fail"] == 35, f"Dev A denom mismatch: {summ_dev_A['total_n']}/{summ_dev_A['total_fail']} vs 879/35"
assert summ_dev_A["covered_fail"] == int(wA["covered_fail"]), f"Dev A covered_fail mismatch: {summ_dev_A['covered_fail']} vs {wA['covered_fail']}"
assert summ_dev_A["covered_n"] == int(wA["covered_n"]), f"Dev A covered_n mismatch: {summ_dev_A['covered_n']} vs {wA['covered_n']}"
assert np.allclose(summ_dev_A["coverage"], float(wA["coverage"]), atol=1e-8, rtol=1e-6), "Dev A coverage mismatch!"
assert np.allclose(summ_dev_A["workload"], float(wA["workload"]), atol=1e-8, rtol=1e-6), "Dev A workload mismatch!"

assert summ_dev_B["total_n"] == 879 and summ_dev_B["total_fail"] == 35, f"Dev B denom mismatch: {summ_dev_B['total_n']}/{summ_dev_B['total_fail']} vs 879/35"
assert summ_dev_B["covered_fail"] == int(wB["covered_fail"]), f"Dev B covered_fail mismatch: {summ_dev_B['covered_fail']} vs {wB['covered_fail']}"
assert summ_dev_B["covered_n"] == int(wB["covered_n"]), f"Dev B covered_n mismatch: {summ_dev_B['covered_n']} vs {wB['covered_n']}"
assert np.allclose(summ_dev_B["coverage"], float(wB["coverage"]), atol=1e-8, rtol=1e-6), "Dev B coverage mismatch!"
assert np.allclose(summ_dev_B["workload"], float(wB["workload"]), atol=1e-8, rtol=1e-6), "Dev B workload mismatch!"

# Budget breaches audit for Branch A across disjoint folds
compact_dev_folds_A = []
for f in fold_dev_A:
    wl = float(f["workload"])
    cov = float(f["coverage"]) if pd.notna(f["coverage"]) else np.nan
    compact_dev_folds_A.append({
        "fold_idx": int(f["fold_idx"]),
        "start": str(f["start"]),
        "end": str(f["end"]),
        "total_n": int(f["total_n"]),
        "total_fail": int(f["total_fail"]),
        "covered_n": int(f["covered_n"]),
        "covered_fail": int(f["covered_fail"]),
        "coverage": cov,
        "workload": wl,
        "precision_entity": float(f.get("precision_entity", np.nan)),
        "breach_nominal": bool(wl > sel_budget_A),
        "breach_30": bool(wl > 0.30)
    })

count_breach_nominal_A = sum(1 for f in compact_dev_folds_A if f["breach_nominal"])
count_breach_30_A = sum(1 for f in compact_dev_folds_A if f["breach_30"])

compact_dev_folds_B = []
for f in fold_dev_B:
    compact_dev_folds_B.append({
        "fold_idx": int(f["fold_idx"]),
        "start": str(f["start"]),
        "end": str(f["end"]),
        "total_n": int(f["total_n"]),
        "total_fail": int(f["total_fail"]),
        "covered_n": int(f["covered_n"]),
        "covered_fail": int(f["covered_fail"]),
        "raw_recall": float(f["coverage"]),
        "workload": float(f["workload"]),
        "precision": float(f.get("precision_entity", np.nan)),
        "quota_ceiling_checked": bool(f["covered_n"] <= int(np.floor(sel_budget_B * f["total_n"])))
    })

compact_dev_report = {
    "branch_A": {
        "candidate": sel_cand_A,
        "global_summary": {
            "coverage": float(summ_dev_A["coverage"]),
            "workload": float(summ_dev_A["workload"]),
            "covered_fail": int(summ_dev_A["covered_fail"]),
            "total_fail": int(summ_dev_A["total_fail"]),
            "covered_n": int(summ_dev_A["covered_n"]),
            "total_n": int(summ_dev_A["total_n"]),
            "precision_entity": float(summ_dev_A.get("precision_entity", np.nan))
        },
        "folds": compact_dev_folds_A,
        "breach_nominal_count": count_breach_nominal_A,
        "breach_30_count": count_breach_30_A
    },
    "branch_B": {
        "candidate": sel_cand_B,
        "global_summary": {
            "raw_recall": float(summ_dev_B["coverage"]),
            "workload": float(summ_dev_B["workload"]),
            "covered_fail": int(summ_dev_B["covered_fail"]),
            "total_fail": int(summ_dev_B["total_fail"]),
            "covered_n": int(summ_dev_B["covered_n"]),
            "total_n": int(summ_dev_B["total_n"]),
            "precision": float(summ_dev_B.get("precision_entity", np.nan))
        },
        "folds": compact_dev_folds_B
    }
}

print("-----------------------------------------------------------------")
print(f"[STEP 1 DEV AUDIT] Branch A ({sel_cand_A}) Dev Replay Folds:")
for f in compact_dev_folds_A:
    print(f"  Fold {f['fold_idx']} [{f['start']} -> {f['end']}]: Coverage={f['coverage']:.4f} ({f['covered_fail']}/{f['total_fail']}), Workload={f['workload']:.4f} ({f['covered_n']}/{f['total_n']}), Precision={f['precision_entity']:.4f}, BreachNominal={f['breach_nominal']}, Breach30%={f['breach_30']}")
print(f"  Total Budget Breaches: >Nominal({sel_budget_A:.2f})={count_breach_nominal_A}/3 folds, >30%={count_breach_30_A}/3 folds")

print(f"[STEP 1 DEV AUDIT] Branch B ({sel_cand_B}) Dev Replay Folds:")
for f in compact_dev_folds_B:
    print(f"  Fold {f['fold_idx']} [{f['start']} -> {f['end']}]: Recall={f['raw_recall']:.4f} ({f['covered_fail']}/{f['total_fail']}), Workload={f['workload']:.4f} ({f['covered_n']}/{f['total_n']}), Precision={f['precision']:.4f}, QuotaOK={f['quota_ceiling_checked']}")
print("-----------------------------------------------------------------")

# 2. Step 2: DEV-ONLY Delay Robustness (D=12, D=24 vs D=6)
# Evaluate on dev eval span [2008-08-21 18:00, 2008-10-02 18:00) with 879 samples, 35 Fail
delay_robustness = {}
delay_robustness[6] = {
    "coverage": float(wA["coverage"]),
    "workload": float(wA["workload"]),
    "covered_fail": int(wA["covered_fail"]),
    "total_fail": int(wA["total_fail"]),
    "covered_n": int(wA["covered_n"]),
    "total_n": int(wA["total_n"]),
    "delay_h": 6
}
assert delay_robustness[6]["total_n"] == 879 and delay_robustness[6]["total_fail"] == 35, f"Expected D=6 Dev denom 879/35, got {delay_robustness[6]['total_n']}/{delay_robustness[6]['total_fail']}"

for test_d in [12, 24]:
    cfg_d = dict(cfg_A)
    cfg_d["delay_h"] = test_d
    raw_sub = raw_df[raw_df["timestamp"] < dev_cutoff].copy()
    orig_min = raw_sub["timestamp"].min() + pd.Timedelta(hours=168)
    orig_start = orig_min.ceil("6h")
    orig_end = raw_sub["timestamp"].max().floor("6h")
    dev_origs = pd.date_range(orig_start, orig_end, freq="6h")
    
    X_d, meta_d = make_features(raw_sub, origins=dev_origs, delay_h=test_d, history_h=168, lead_h=6)
    tgt_d = build_targets(raw_sub, origins=dev_origs, horizon_h=sel_H, lead_h=6, delay_h=test_d, q=0.1, min_event_samples=5)
    tgt_d = tgt_d.set_index("origin", drop=False)
    
    scores_d, _, _ = forecast_stream(raw_sub, X_d, meta_d, tgt_d, sel_cand_A, cfg_d)
    w_df_d, summ_d, _ = apply_forecast_policy(scores_d, raw_sub, tgt_d, budget=sel_budget_A, lookback_days=sel_lookback_A, eval_span=(dev_eval_start, dev_eval_end))
    
    assert summ_d["total_n"] == 879, f"D={test_d} dev total_n mismatch: expected 879, got {summ_d['total_n']}"
    assert summ_d["total_fail"] == 35, f"D={test_d} dev total_fail mismatch: expected 35, got {summ_d['total_fail']}"
    
    delay_robustness[test_d] = {
        "coverage": float(summ_d["coverage"]),
        "workload": float(summ_d["workload"]),
        "covered_fail": int(summ_d["covered_fail"]),
        "total_fail": int(summ_d["total_fail"]),
        "covered_n": int(summ_d["covered_n"]),
        "total_n": int(summ_d["total_n"]),
        "delay_h": test_d
    }

print(f"[STEP 2] Delay Robustness on Dev (Asserted Denom 879/35): {delay_robustness}")


[STEP 1] Loaded V2 artifact keys: ['protocol', 'stream_configs', 'grids', 'raw_hashes', 'temporal_bounds', 'forecast_summary', 'qc_summary', 'selection', 'selected_dev_scores', 'selected_last_states', 'selected_traces', 'versions', 'policy_refinement_v2']


[STEP 1] Raw data SHA256 matches artifact perfectly.
[STEP 1] Environment & Source Code Hashes: {'02_preprocessing_code_sha256': '23729b8e06991e63410740b9d6c5bf518051b403d9d6c8f424cc23cb7254b052', '03_training_helpers_code_sha256': '114bc564530dfab11e4d25d3fad07e99155f609863b6cc454c7f6e824adeec85', 'numpy_version': '2.5.4', 'pandas_version': '3.0.6', 'sklearn_version': '1.9.1', 'prior_holdout_known_exposure': True, 'client_date': '2026-10-26 Asia/Saigon'}
[STEP 1] Selected DEV Winner A: Candidate=ridge_history, H=6h, Budget=0.30, Lookback=7d
  Coverage: 0.3429 (12/35), Workload: 0.2514 (221/879)
[STEP 1] Selected DEV Winner B: Candidate=balanced_hgb, QuotaBudget=0.30, ThresholdBudget=0.40, Lookback=14d
  Raw Recall: 0.4571 (16/35), Workload: 0.2696 (237/879)


-----------------------------------------------------------------
[STEP 1 DEV AUDIT] Branch A (ridge_history) Dev Replay Folds:
  Fold 0 [2008-08-21 18:00:00 -> 2008-09-04 18:00:00]: Coverage=0.3333 (5/15), Workload=0.2762 (100/362), Precision=0.0500, BreachNominal=False, Breach30%=False
  Fold 1 [2008-09-04 18:00:00 -> 2008-09-18 18:00:00]: Coverage=0.4000 (2/5), Workload=0.3137 (48/153), Precision=0.0417, BreachNominal=True, Breach30%=True
  Fold 2 [2008-09-18 18:00:00 -> 2008-10-02 18:00:00]: Coverage=0.3333 (5/15), Workload=0.2005 (73/364), Precision=0.0685, BreachNominal=False, Breach30%=False
  Total Budget Breaches: >Nominal(0.30)=1/3 folds, >30%=1/3 folds
[STEP 1 DEV AUDIT] Branch B (balanced_hgb) Dev Replay Folds:
  Fold 0 [2008-08-21 18:00:00 -> 2008-09-04 18:00:00]: Recall=0.4667 (7/15), Workload=0.2348 (85/362), Precision=0.0824, QuotaOK=True
  Fold 1 [2008-09-04 18:00:00 -> 2008-09-18 18:00:00]: Recall=0.2000 (1/5), Workload=0.2941 (45/153), Precision=0.0222, QuotaOK=True


[STEP 2] Delay Robustness on Dev (Asserted Denom 879/35): {6: {'coverage': 0.34285714285714286, 'workload': 0.2514220705346985, 'covered_fail': 12, 'total_fail': 35, 'covered_n': 221, 'total_n': 879, 'delay_h': 6}, 12: {'coverage': 0.2571428571428571, 'workload': 0.2241183162684869, 'covered_fail': 9, 'total_fail': 35, 'covered_n': 197, 'total_n': 879, 'delay_h': 12}, 24: {'coverage': 0.4, 'workload': 0.27986348122866894, 'covered_fail': 14, 'total_fail': 35, 'covered_n': 246, 'total_n': 879, 'delay_h': 24}}


In [3]:
# Cell 3: Step 3 (Full Forecast Stream & Dev Reproduction) & Step 4 (Branch A Holdout Evaluation)
# 1. Full feature grid with D=6
raw_min = raw_df["timestamp"].min() + pd.Timedelta(hours=168)
full_orig_start = raw_min.ceil("6h")
full_orig_end = raw_df["timestamp"].max().floor("6h")
full_origins = pd.date_range(full_orig_start, full_orig_end, freq="6h")

X_full, meta_full = make_features(raw_df, origins=full_origins, delay_h=6, history_h=168, lead_h=6)
tgt_full = build_targets(raw_df, origins=full_origins, horizon_h=sel_H, lead_h=6, delay_h=6, q=0.1, min_event_samples=5)
tgt_full = tgt_full.set_index("origin", drop=False)

scores_full_A, last_state_A, trace_A = forecast_stream(raw_df, X_full, meta_full, tgt_full, sel_cand_A, cfg_A)

# Assert causal timestamps
valid_fit = scores_full_A["fit_time"].dropna()
valid_orig = scores_full_A.loc[valid_fit.index, "origin"]
assert (valid_fit <= valid_orig).all(), "Causality violation: fit_time > origin!"
if "last_train_label_available" in scores_full_A.columns:
    v_lbl = scores_full_A["last_train_label_available"].dropna()
    assert (v_lbl <= scores_full_A.loc[v_lbl.index, "fit_time"]).all(), "Causality violation: last_train_label_available > fit_time!"
if "feature_max_available" in scores_full_A.columns:
    v_feat = scores_full_A["feature_max_available"].dropna()
    assert (v_feat <= scores_full_A.loc[v_feat.index, "origin"]).all(), "Causality violation: feature_max_available > origin!"

# Assert reproduced dev scores match saved dev scores exactly
saved_scores_A = exp["selected_dev_scores"]["branch_A"]
common_idx = saved_scores_A.index
reproduced_dev_scores = scores_full_A.loc[common_idx, "score"].values
saved_dev_scores = saved_scores_A["score"].values
np.testing.assert_allclose(reproduced_dev_scores, saved_dev_scores, atol=1e-8, rtol=1e-6, equal_nan=True)
print("[STEP 3] Full causal stream reproduces saved Dev scores on all common origins perfectly.")

# 2. Step 4: Evaluate Branch A on Holdout Test A [2008-10-03 06:00, 2008-10-17 06:00)
test_A_start = pd.Timestamp(exp["temporal_bounds"]["test_A_start"])
test_A_end = pd.Timestamp(exp["temporal_bounds"]["test_A_end"])

warn_df_A, summary_A, fold_A = apply_forecast_policy(
    scores_full_A, raw_df, tgt_full,
    budget=sel_budget_A,
    lookback_days=sel_lookback_A,
    eval_span=(test_A_start, test_A_end)
)

raw_test_A = raw_df[(raw_df["timestamp"] >= test_A_start) & (raw_df["timestamp"] < test_A_end)]
assert len(raw_test_A) == int(exp["temporal_bounds"]["test_A_expected_n"]), f"Expected {exp['temporal_bounds']['test_A_expected_n']} rows, got {len(raw_test_A)}"
assert raw_test_A["fail"].sum() == int(exp["temporal_bounds"]["test_A_expected_fail"]), f"Expected {exp['temporal_bounds']['test_A_expected_fail']} fails, got {raw_test_A['fail'].sum()}"

# Total holdout accounting across 314
cutoff_ts = pd.Timestamp(exp["protocol"]["dev_cutoff"])
raw_314 = raw_df[raw_df["timestamp"] >= cutoff_ts]
raw_pre = raw_df[(raw_df["timestamp"] >= cutoff_ts) & (raw_df["timestamp"] < test_A_start)]
raw_post = raw_df[raw_df["timestamp"] >= test_A_end]
assert len(raw_314) == 314 and raw_314["fail"].sum() == 17, "Holdout 314 accounting mismatch!"
assert len(raw_pre) == 15 and raw_pre["fail"].sum() == 1, "Pre-span accounting mismatch (expected 15 rows, 1 fail)!"
assert len(raw_post) == 2 and raw_post["fail"].sum() == 0, "Post-span accounting mismatch (expected 2 rows, 0 fail)!"

# Strictly filter valid evaluation windows: target_start >= test_A_start AND target_end <= test_A_end AND origin >= test_A_start - 6h
valid_eval_mask_A = (
    (warn_df_A["target_start"] >= test_A_start) &
    (warn_df_A["target_end"] <= test_A_end) &
    (warn_df_A["origin"] >= (test_A_start - pd.Timedelta(hours=6)))
)
eval_origins_A = warn_df_A.loc[valid_eval_mask_A].copy().reset_index(drop=True)

# Explicit 1-to-1 inner join on origin between scores and target events/rates
score_slice_A = eval_origins_A[["origin", "score"]].copy()
tgt_slice_A = tgt_full[["origin", "target_start", "target_end", "event", "rate", "n_fail", "n_future"]].reset_index(drop=True)
aligned_eval_A = score_slice_A.merge(tgt_slice_A, on="origin", how="inner", validate="one_to_one")
assert len(aligned_eval_A) == len(eval_origins_A), "Aligned eval rows mismatch with eval_origins_A!"

# Event ROC-AUC and AP on defined binary events
valid_event_mask = aligned_eval_A["event"].isin([0, 1]) & np.isfinite(aligned_eval_A["score"])
if valid_event_mask.sum() > 0 and len(np.unique(aligned_eval_A.loc[valid_event_mask, "event"])) > 1:
    ev_true = aligned_eval_A.loc[valid_event_mask, "event"].values.astype(int)
    ev_score = aligned_eval_A.loc[valid_event_mask, "score"].values
    event_roc_auc = float(roc_auc_score(ev_true, ev_score))
    event_ap = float(average_precision_score(ev_true, ev_score))
else:
    event_roc_auc = np.nan
    event_ap = np.nan

# Rate regressor/baseline MAE & RMSE against nonempty realized rate
nonempty_rate_mask = aligned_eval_A["rate"].dropna()
regressor_rate_models = [
    "fail_rate24", "fail_rate72", "fail_rate168", "past_qc_ewma24",
    "ridge_history", "ridge_sensor16", "et_rate24", "hgb_poisson24"
]
is_regressor_or_rate = sel_cand_A in regressor_rate_models
if is_regressor_or_rate and len(nonempty_rate_mask) > 0:
    sub_rate = aligned_eval_A.loc[nonempty_rate_mask.index]
    valid_rate_pair = np.isfinite(sub_rate["rate"]) & np.isfinite(sub_rate["score"])
    if valid_rate_pair.sum() > 0:
        rate_true = sub_rate.loc[valid_rate_pair, "rate"].values
        rate_pred = sub_rate.loc[valid_rate_pair, "score"].values
        rate_mae = float(mean_absolute_error(rate_true, rate_pred))
        rate_rmse = float(np.sqrt(mean_squared_error(rate_true, rate_pred)))
    else:
        rate_mae, rate_rmse = np.nan, np.nan
else:
    rate_mae, rate_rmse = np.nan, np.nan

summary_A["event_roc_auc"] = event_roc_auc
summary_A["event_average_precision"] = event_ap
summary_A["rate_mae"] = rate_mae
summary_A["rate_rmse"] = rate_rmse

# Three pre-declared baselines
baselines_A = {}
for base_name in ["fail_rate24", "fail_rate72", "fail_rate168"]:
    base_scores, _, _ = forecast_stream(raw_df, X_full, meta_full, tgt_full, base_name, cfg_A)
    _, base_summ, _ = apply_forecast_policy(base_scores, raw_df, tgt_full, budget=sel_budget_A, lookback_days=sel_lookback_A, eval_span=(test_A_start, test_A_end))
    assert base_summ["total_n"] == 297 and base_summ["total_fail"] == 16, f"Baseline {base_name} denom mismatch: expected 297/16, got {base_summ['total_n']}/{base_summ['total_fail']}"
    baselines_A[base_name] = {
        "coverage": float(base_summ["coverage"]),
        "workload": float(base_summ["workload"]),
        "covered_fail": int(base_summ["covered_fail"]),
        "total_fail": int(base_summ["total_fail"]),
        "covered_n": int(base_summ["covered_n"]),
        "total_n": int(base_summ["total_n"]),
        "precision_entity": float(base_summ.get("precision_entity", np.nan))
    }

print(f"[STEP 4] Branch A ({sel_cand_A}) Summary on Test A (297 rows, 16 Fails):")
print(f"  Raw Fail Coverage: {summary_A['coverage']:.4f} ({summary_A['covered_fail']}/{summary_A['total_fail']})")
print(f"  Raw Workload:      {summary_A['workload']:.4f} ({summary_A['covered_n']}/{summary_A['total_n']})")
print(f"  Entity Precision:  {summary_A.get('precision_entity', np.nan):.4f}")
print(f"  Time Under Alert:  {summary_A.get('time_under_alert', np.nan):.4f}")
print(f"  Window Event Metrics: Precision={summary_A.get('event_precision', np.nan):.4f}, Recall={summary_A.get('event_recall', np.nan):.4f}, ROC-AUC={event_roc_auc}, AP={event_ap}")
print(f"  Rate Metrics (Regressor only): MAE={rate_mae}, RMSE={rate_rmse}")
print(f"[STEP 4] Pre-declared Baselines comparison (Denom 297/16): {baselines_A}")


[STEP 3] Full causal stream reproduces saved Dev scores on all common origins perfectly.


[STEP 4] Branch A (ridge_history) Summary on Test A (297 rows, 16 Fails):
  Raw Fail Coverage: 0.5625 (9/16)
  Raw Workload:      0.4478 (133/297)
  Entity Precision:  0.0677
  Time Under Alert:  0.4821
  Window Event Metrics: Precision=0.3571, Recall=0.7143, ROC-AUC=0.5170068027210885, AP=0.28566455822094916
  Rate Metrics (Regressor only): MAE=0.09887927682344365, RMSE=0.12273614897190663
[STEP 4] Pre-declared Baselines comparison (Denom 297/16): {'fail_rate24': {'coverage': 0.25, 'workload': 0.2760942760942761, 'covered_fail': 4, 'total_fail': 16, 'covered_n': 82, 'total_n': 297, 'precision_entity': 0.04878048780487805}, 'fail_rate72': {'coverage': 0.3125, 'workload': 0.4107744107744108, 'covered_fail': 5, 'total_fail': 16, 'covered_n': 122, 'total_n': 297, 'precision_entity': 0.040983606557377046}, 'fail_rate168': {'coverage': 0.8125, 'workload': 0.5387205387205387, 'covered_fail': 13, 'total_fail': 16, 'covered_n': 160, 'total_n': 297, 'precision_entity': 0.08125}}


In [4]:
# Cell 4: Step 5 (QC Stream & Branch B Holdout Evaluation) & Step 6 (Independent Verification)
scores_full_B, last_state_B, trace_B = qc_stream(raw_df, sel_cand_B, cfg_B)

# Verify Dev score reproduction for Branch B via unique raw_row_id
saved_scores_B = exp["selected_dev_scores"]["branch_B"].set_index("raw_row_id")
scores_full_B_indexed = scores_full_B.set_index("raw_row_id")
common_b_ids = saved_scores_B.index
np.testing.assert_allclose(
    scores_full_B_indexed.loc[common_b_ids, "score"].values,
    saved_scores_B.loc[common_b_ids, "score"].values,
    atol=1e-8, rtol=1e-6, equal_nan=True
)
print("[STEP 5] Branch B full QC stream reproduces saved Dev predictions on all common raw IDs.")

# Causality assertions for B
valid_fit_B = scores_full_B["fit_time"].dropna()
v_ts_B = scores_full_B.loc[valid_fit_B.index, "timestamp"]
assert (valid_fit_B <= v_ts_B).all(), "Causality violation in B: fit_time > timestamp!"
if "last_train_label_available" in scores_full_B.columns:
    v_lbl_B = scores_full_B["last_train_label_available"].dropna()
    assert (v_lbl_B <= scores_full_B.loc[v_lbl_B.index, "fit_time"]).all(), "Causality violation in B: last_train_label_available > fit_time!"

# Evaluate Branch B on entire 314 holdout [cutoff, raw_max + 1ns)
eval_span_B = [(cutoff_ts, raw_df["timestamp"].max() + pd.Timedelta(nanoseconds=1))]
warn_df_B, summary_B, fold_B = apply_qc_policy(
    scores_full_B, raw_df,
    budget=sel_budget_B,
    threshold_budget=sel_threshold_budget_B,
    lookback_days=sel_lookback_B,
    report_spans=eval_span_B
)

# Extract holdout slice for B
holdout_warn_B = warn_df_B[warn_df_B["timestamp"] >= cutoff_ts].copy()
assert len(holdout_warn_B) == 314, f"Expected 314 rows in B holdout, got {len(holdout_warn_B)}"
assert holdout_warn_B["raw_row_id"].is_unique, "raw_row_id is not unique in B holdout!"

# Hard quota assertion: final_warn count <= floor(budget * N)
quota_limit = int(np.floor(sel_budget_B * 314))
actual_warn_count_B = holdout_warn_B["final_warn"].sum()
assert actual_warn_count_B <= quota_limit, f"Quota violated: {actual_warn_count_B} > {quota_limit}"

# Merge with truth ONLY for offline evaluation using validate='one_to_one'
holdout_eval_B = holdout_warn_B.reset_index(drop=True).merge(
    raw_df[["raw_row_id", "fail"]].reset_index(drop=True),
    on="raw_row_id",
    how="left",
    validate="one_to_one"
)
y_true_B = holdout_eval_B["fail"].values
y_pred_B = holdout_eval_B["final_warn"].astype(int).values
y_score_B = holdout_eval_B["score"].values

assert y_true_B.sum() == 17, f"Expected 17 fails in B holdout, got {y_true_B.sum()}"

tn, fp, fn, tp = confusion_matrix(y_true_B, y_pred_B, labels=[0, 1]).ravel()
assert tp + fn == 17, f"Expected TP+FN=17, got {tp+fn}"
assert tn + fp == 297, f"Expected TN+FP=297, got {tn+fp}"
assert tp + fn + tn + fp == 314, f"Expected total 314, got {tp+fn+tn+fp}"

prec_val_B = precision_score(y_true_B, y_pred_B, zero_division=np.nan) if y_pred_B.sum() > 0 else np.nan

metrics_B = {
    "recall": recall_score(y_true_B, y_pred_B, zero_division=0),
    "precision": prec_val_B,
    "f1": f1_score(y_true_B, y_pred_B, zero_division=0),
    "accuracy": accuracy_score(y_true_B, y_pred_B),
    "balanced_accuracy": balanced_accuracy_score(y_true_B, y_pred_B),
    "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
    "workload": float(actual_warn_count_B / 314),
    "quota_suppressed_count": int(holdout_eval_B["quota_suppressed"].sum()) if "quota_suppressed" in holdout_eval_B else 0,
    "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
    "total_n": 314, "total_fail": 17
}

finite_m_B = np.isfinite(y_score_B)
if finite_m_B.sum() > 0 and len(np.unique(y_true_B[finite_m_B])) > 1:
    metrics_B["roc_auc"] = float(roc_auc_score(y_true_B[finite_m_B], y_score_B[finite_m_B]))
    metrics_B["average_precision"] = float(average_precision_score(y_true_B[finite_m_B], y_score_B[finite_m_B]))
else:
    metrics_B["roc_auc"] = np.nan
    metrics_B["average_precision"] = np.nan

# Explicit comparison between metrics_B and summary_B
assert metrics_B["TP"] == summary_B["covered_fail"], f"TP mismatch with summary_B: {metrics_B['TP']} vs {summary_B['covered_fail']}"
assert metrics_B["TP"] + metrics_B["FP"] == summary_B["covered_n"], f"Covered_n mismatch: {metrics_B['TP']+metrics_B['FP']} vs {summary_B['covered_n']}"

print(f"[STEP 5] Branch B ({sel_cand_B}) Metrics on Holdout 314:")
print(f"  Raw Recall:  {metrics_B['recall']:.4f} ({metrics_B['TP']}/{metrics_B['TP']+metrics_B['FN']})")
print(f"  Workload:    {metrics_B['workload']:.4f} ({actual_warn_count_B}/314)")
print(f"  Precision:   {metrics_B['precision']:.4f}, Specificity: {metrics_B['specificity']:.4f}, F1: {metrics_B['f1']:.4f}")
print(f"  Confusion Matrix: TP={tp}, FP={fp}, FN={fn}, TN={tn} (Total=314, Fails=17)")

# Step 6: Independent Audit Verification
# A: Recompute incidence matrix & union strictly using valid evaluation window
inc_A = incidence(raw_test_A, eval_origins_A)
active_warns_A = eval_origins_A["warn"].values.astype(bool)

if active_warns_A.any():
    recomp_cov_mask_A = inc_A[active_warns_A].any(axis=0)
else:
    recomp_cov_mask_A = np.zeros(len(raw_test_A), dtype=bool)

recomp_cov_fail_A = int(raw_test_A.loc[recomp_cov_mask_A, "fail"].sum())
recomp_cov_n_A = int(recomp_cov_mask_A.sum())

assert recomp_cov_fail_A == summary_A["covered_fail"], f"Recomputed Fail mismatch: {recomp_cov_fail_A} vs {summary_A['covered_fail']}"
assert recomp_cov_n_A == summary_A["covered_n"], f"Recomputed Covered_n mismatch: {recomp_cov_n_A} vs {summary_A['covered_n']}"
assert summary_A["total_n"] == 297 and summary_A["total_fail"] == 16, f"Summary A denom mismatch: expected 297/16, got {summary_A['total_n']}/{summary_A['total_fail']}"
print("[STEP 6] Independent incidence union audit for Branch A passed (Both covered_n and covered_fail match summary_A exactly).")


[STEP 5] Branch B full QC stream reproduces saved Dev predictions on all common raw IDs.


[STEP 5] Branch B (balanced_hgb) Metrics on Holdout 314:
  Raw Recall:  0.2941 (5/17)
  Workload:    0.2739 (86/314)
  Precision:   0.0581, Specificity: 0.7273, F1: 0.0971
  Confusion Matrix: TP=5, FP=81, FN=12, TN=216 (Total=314, Fails=17)
[STEP 6] Independent incidence union audit for Branch A passed (Both covered_n and covered_fail match summary_A exactly).


In [5]:
# Cell 5: Step 7 (Two-Day Calendar Block Bootstrap & Daily Breakdown)
# 1. 2-Day Calendar Block Bootstrap for Branch A
rng = np.random.RandomState(42)
test_A_duration = test_A_end - test_A_start
n_blocks_A = int(np.ceil(test_A_duration / pd.Timedelta(days=2)))
block_sec = 2 * 86400

raw_test_A_blocks = raw_test_A.copy()
raw_test_A_blocks["sec"] = (raw_test_A_blocks["timestamp"] - test_A_start).dt.total_seconds()
raw_test_A_blocks["block_id"] = (raw_test_A_blocks["sec"] // block_sec).astype(int)
raw_test_A_blocks["warned"] = recomp_cov_mask_A

block_stats_A = []
for b in range(n_blocks_A):
    sub = raw_test_A_blocks[raw_test_A_blocks["block_id"] == b]
    block_stats_A.append({
        "n": len(sub),
        "fail": int(sub["fail"].sum()),
        "covered_n": int(sub["warned"].sum()),
        "covered_fail": int((sub["warned"] & (sub["fail"] == 1)).sum())
    })

# Assert blocks reconstruct totals exactly
assert sum(b["n"] for b in block_stats_A) == 297, f"Block reconstruction total_n mismatch: {sum(b['n'] for b in block_stats_A)} vs 297"
assert sum(b["fail"] for b in block_stats_A) == 16, f"Block reconstruction fail mismatch: {sum(b['fail'] for b in block_stats_A)} vs 16"

boot_cov_A = []
boot_wl_A = []
for _ in range(1000):
    sample_b = rng.choice(n_blocks_A, size=n_blocks_A, replace=True)
    tot_f = sum(block_stats_A[b]["fail"] for b in sample_b)
    tot_n = sum(block_stats_A[b]["n"] for b in sample_b)
    cov_f = sum(block_stats_A[b]["covered_fail"] for b in sample_b)
    cov_n = sum(block_stats_A[b]["covered_n"] for b in sample_b)
    if tot_f > 0:
        boot_cov_A.append(cov_f / tot_f)
    if tot_n > 0:
        boot_wl_A.append(cov_n / tot_n)

ci_A = {
    "coverage_2.5": float(np.percentile(boot_cov_A, 2.5)) if boot_cov_A else np.nan,
    "coverage_97.5": float(np.percentile(boot_cov_A, 97.5)) if boot_cov_A else np.nan,
    "workload_2.5": float(np.percentile(boot_wl_A, 2.5)) if boot_wl_A else np.nan,
    "workload_97.5": float(np.percentile(boot_wl_A, 97.5)) if boot_wl_A else np.nan,
    "valid_reps": len(boot_cov_A),
    "uncertainty_note": "Conditional frozen replay on small N (16 Fail); reflects observation cluster variability, not full model selection uncertainty."
}
print(f"[STEP 7] Branch A 2-Day Block Bootstrap 95% CI: {ci_A}")

# 2. 2-Day Calendar Block Bootstrap for Branch B
b_start = cutoff_ts
b_end = raw_df["timestamp"].max() + pd.Timedelta(nanoseconds=1)
b_duration = b_end - b_start
n_blocks_B = int(np.ceil(b_duration / pd.Timedelta(days=2)))

holdout_eval_B["sec"] = (holdout_eval_B["timestamp"] - b_start).dt.total_seconds()
holdout_eval_B["block_id"] = (holdout_eval_B["sec"] // block_sec).astype(int)

block_stats_B = []
for b in range(n_blocks_B):
    sub = holdout_eval_B[holdout_eval_B["block_id"] == b]
    block_stats_B.append({
        "n": len(sub),
        "fail": int(sub["fail"].sum()),
        "tp": int(((sub["final_warn"] == 1) & (sub["fail"] == 1)).sum()),
        "warned": int((sub["final_warn"] == 1).sum())
    })

assert sum(b["n"] for b in block_stats_B) == 314, f"Block B reconstruction total_n mismatch: {sum(b['n'] for b in block_stats_B)} vs 314"
assert sum(b["fail"] for b in block_stats_B) == 17, f"Block B reconstruction fail mismatch: {sum(b['fail'] for b in block_stats_B)} vs 17"

boot_rec_B = []
boot_wl_B = []
for _ in range(1000):
    sample_b = rng.choice(n_blocks_B, size=n_blocks_B, replace=True)
    tot_f = sum(block_stats_B[b]["fail"] for b in sample_b)
    tot_n = sum(block_stats_B[b]["n"] for b in sample_b)
    tp = sum(block_stats_B[b]["tp"] for b in sample_b)
    warned = sum(block_stats_B[b]["warned"] for b in sample_b)
    if tot_f > 0:
        boot_rec_B.append(tp / tot_f)
    if tot_n > 0:
        boot_wl_B.append(warned / tot_n)

ci_B = {
    "recall_2.5": float(np.percentile(boot_rec_B, 2.5)) if boot_rec_B else np.nan,
    "recall_97.5": float(np.percentile(boot_rec_B, 97.5)) if boot_rec_B else np.nan,
    "workload_2.5": float(np.percentile(boot_wl_B, 2.5)) if boot_wl_B else np.nan,
    "workload_97.5": float(np.percentile(boot_wl_B, 97.5)) if boot_wl_B else np.nan,
    "valid_reps": len(boot_rec_B),
    "uncertainty_note": "Conditional frozen replay on small N (17 Fail); reflects observation cluster variability, not full model selection uncertainty."
}
print(f"[STEP 7] Branch B 2-Day Block Bootstrap 95% CI: {ci_B}")

# Daily Breakdown for Branch A (Reindexed across calendar days strictly intersecting [test_A_start, test_A_end))
all_days_A = pd.date_range(test_A_start.floor("D"), (test_A_end - pd.Timedelta(nanoseconds=1)).floor("D"), freq="D").date
raw_test_A_blocks["date"] = raw_test_A_blocks["timestamp"].dt.date
daily_A_agg = raw_test_A_blocks.groupby("date").apply(
    lambda g: pd.Series({
        "total_n": len(g),
        "total_fail": int(g["fail"].sum()),
        "covered_fail": int((g["warned"] & (g["fail"] == 1)).sum()),
        "covered_n": int(g["warned"].sum()),
        "coverage": ((g["warned"] & (g["fail"] == 1)).sum() / g["fail"].sum()) if g["fail"].sum() > 0 else np.nan,
        "workload": (g["warned"].sum() / len(g)) if len(g) > 0 else np.nan
    })
)
daily_A = daily_A_agg.reindex(all_days_A)
daily_A["total_n"] = daily_A["total_n"].fillna(0).astype(int)
daily_A["total_fail"] = daily_A["total_fail"].fillna(0).astype(int)
daily_A["covered_fail"] = daily_A["covered_fail"].fillna(0).astype(int)
daily_A["covered_n"] = daily_A["covered_n"].fillna(0).astype(int)

# Daily Breakdown for Branch B (Reindexed across calendar days strictly intersecting [cutoff_ts, b_end))
all_days_B = pd.date_range(cutoff_ts.floor("D"), (b_end - pd.Timedelta(nanoseconds=1)).floor("D"), freq="D").date
holdout_eval_B["date"] = holdout_eval_B["timestamp"].dt.date
daily_B_agg = holdout_eval_B.groupby("date").apply(
    lambda g: pd.Series({
        "total_n": len(g),
        "total_fail": int(g["fail"].sum()),
        "tp": int(((g["final_warn"] == 1) & (g["fail"] == 1)).sum()),
        "warned": int((g["final_warn"] == 1).sum()),
        "recall": (((g["final_warn"] == 1) & (g["fail"] == 1)).sum() / g["fail"].sum()) if g["fail"].sum() > 0 else np.nan,
        "workload": ((g["final_warn"] == 1).sum() / len(g)) if len(g) > 0 else np.nan
    })
)
daily_B = daily_B_agg.reindex(all_days_B)
daily_B["total_n"] = daily_B["total_n"].fillna(0).astype(int)
daily_B["total_fail"] = daily_B["total_fail"].fillna(0).astype(int)
daily_B["tp"] = daily_B["tp"].fillna(0).astype(int)
daily_B["warned"] = daily_B["warned"].fillna(0).astype(int)

print(f"[STEP 7] Complete Daily Tables created: Branch A has {len(daily_A)} days, Branch B has {len(daily_B)} days.")


[STEP 7] Branch A 2-Day Block Bootstrap 95% CI: {'coverage_2.5': 0.0, 'coverage_97.5': 0.7647058823529411, 'workload_2.5': 0.17133956386292834, 'workload_97.5': 0.7004172127566621, 'valid_reps': 1000, 'uncertainty_note': 'Conditional frozen replay on small N (16 Fail); reflects observation cluster variability, not full model selection uncertainty.'}
[STEP 7] Branch B 2-Day Block Bootstrap 95% CI: {'recall_2.5': 0.16666666666666666, 'recall_97.5': 0.8333333333333334, 'workload_2.5': 0.22306603773584907, 'workload_97.5': 0.31807246177475185, 'valid_reps': 1000, 'uncertainty_note': 'Conditional frozen replay on small N (17 Fail); reflects observation cluster variability, not full model selection uncertainty.'}
[STEP 7] Complete Daily Tables created: Branch A has 15 days, Branch B has 16 days.


In [6]:
# Cell 6: Step 8 (Prospective 3-Level Decisions & Separate Post-Target Outcome Audit)
# 1. Branch A Prospective Decisions
# Restrict strictly to valid test window: target_start >= test_A_start, target_end <= test_A_end, origin >= test_A_start - 6h
valid_window_mask_A = (
    (warn_df_A["target_start"] >= test_A_start) &
    (warn_df_A["target_end"] <= test_A_end) &
    (warn_df_A["origin"] >= (test_A_start - pd.Timedelta(hours=6)))
)
test_decisions_A = warn_df_A.loc[valid_window_mask_A].copy().reset_index(drop=True)

# Compute causal monitor threshold
tau_track_A_list = []
for orig in test_decisions_A["origin"]:
    tau_trk, _ = capacity_threshold(scores_full_A, raw_df, orig, budget=0.50, lookback_days=sel_lookback_A)
    tau_track_A_list.append(tau_trk)
test_decisions_A["tau_monitor"] = tau_track_A_list

def assign_level_A(r):
    if r["warn"]:
        return "Nguy cơ cao"
    elif np.isfinite(r["score"]) and np.isfinite(r["tau_monitor"]) and r["score"] >= r["tau_monitor"]:
        return "Cần theo dõi"
    else:
        return "Bình thường"

def assign_action_A(lvl):
    if lvl == "Nguy cơ cao":
        return "Kỹ sư chất lượng đánh giá ưu tiên và kiểm tra tín hiệu cảnh báo quy trình"
    elif lvl == "Cần theo dõi":
        return "Tăng cường theo dõi nhật ký cảm biến và chú ý kết quả kiểm định"
    else:
        return "Duy trì quy trình kiểm tra chất lượng (QC) tiêu chuẩn"

test_decisions_A["decision_level"] = test_decisions_A.apply(assign_level_A, axis=1)
test_decisions_A["engineering_action"] = test_decisions_A["decision_level"].apply(assign_action_A)

decisions_A_table = test_decisions_A[[
    "origin", "target_start", "target_end", "target_available",
    "score", "tau", "tau_monitor", "warn", "decision_level", "engineering_action"
]].copy().reset_index(drop=True)

# 2. Branch B Prospective Decisions
tau_track_B_list = []
for ts in holdout_warn_B["timestamp"]:
    tau_trk, _ = qc_threshold(scores_full_B, ts, budget=0.50, lookback_days=sel_lookback_B)
    tau_track_B_list.append(tau_trk)
holdout_warn_B["tau_monitor"] = tau_track_B_list

def assign_level_B(r):
    if r["final_warn"]:
        return "Nguy cơ cao"
    elif (np.isfinite(r["score"]) and np.isfinite(r["tau_monitor"]) and r["score"] >= r["tau_monitor"]) or r.get("quota_suppressed", False):
        return "Cần theo dõi"
    else:
        return "Bình thường"

def assign_action_B(lvl):
    if lvl == "Nguy cơ cao":
        return "Kỹ sư chất lượng đánh giá ưu tiên và tái kiểm tra mẫu"
    elif lvl == "Cần theo dõi":
        return "Theo dõi tín hiệu đo kiểm và ghi nhận trạng thái kiểm tra"
    else:
        return "Duy trì quy trình kiểm tra chất lượng (QC) tiêu chuẩn"

holdout_warn_B["decision_level"] = holdout_warn_B.apply(assign_level_B, axis=1)
holdout_warn_B["engineering_action"] = holdout_warn_B["decision_level"].apply(assign_action_B)

decisions_B_table = holdout_warn_B[[
    "timestamp", "raw_row_id", "score", "tau", "tau_monitor",
    "candidate_high", "final_warn", "quota_suppressed", "decision_level", "engineering_action"
]].copy().reset_index(drop=True)

# Separate Matured Outcome Audit
# Declared audit_asof_A: test_A_end + 6h (strictly bounded to valid evaluation span)
audit_asof_A = test_A_end + pd.Timedelta(hours=6)
matured_audit_A = decisions_A_table[decisions_A_table["target_available"] <= audit_asof_A].copy().reset_index(drop=True)
tgt_slice_A = tgt_full[["origin", "n_fail", "rate", "event"]].reset_index(drop=True)
matured_audit_A = matured_audit_A.merge(tgt_slice_A, on="origin", how="left", validate="one_to_one")

# Declared audit_asof_B: raw_max + 6h
audit_asof_B = raw_df["timestamp"].max() + pd.Timedelta(hours=6)
matured_audit_B = decisions_B_table.copy().reset_index(drop=True)
matured_audit_B["qc_available"] = matured_audit_B["timestamp"] + pd.Timedelta(hours=6)
raw_labels_slice = raw_df[["raw_row_id", "fail"]].reset_index(drop=True)
matured_audit_B = matured_audit_B[matured_audit_B["qc_available"] <= audit_asof_B].merge(
    raw_labels_slice, on="raw_row_id", how="left", validate="one_to_one"
)

print("[STEP 8] Branch A Prospective Decisions (First 12 Test decisions):")
print(decisions_A_table[["origin", "target_start", "target_end", "score", "tau", "decision_level"]].head(12))
print("[STEP 8] Branch B Prospective Decisions Sample (First 8 rows):")
print(decisions_B_table[["timestamp", "raw_row_id", "score", "tau", "decision_level"]].head(8))


[STEP 8] Branch A Prospective Decisions (First 12 Test decisions):
                origin        target_start          target_end     score  \
0  2008-10-03 00:00:00 2008-10-03 06:00:00 2008-10-03 12:00:00  0.025942   
1  2008-10-03 06:00:00 2008-10-03 12:00:00 2008-10-03 18:00:00  0.062149   
2  2008-10-03 12:00:00 2008-10-03 18:00:00 2008-10-04 00:00:00  0.105800   
3  2008-10-03 18:00:00 2008-10-04 00:00:00 2008-10-04 06:00:00  0.096040   
4  2008-10-04 00:00:00 2008-10-04 06:00:00 2008-10-04 12:00:00  0.054259   
5  2008-10-04 06:00:00 2008-10-04 12:00:00 2008-10-04 18:00:00  0.078071   
6  2008-10-04 12:00:00 2008-10-04 18:00:00 2008-10-05 00:00:00  0.115035   
7  2008-10-04 18:00:00 2008-10-05 00:00:00 2008-10-05 06:00:00  0.077677   
8  2008-10-05 00:00:00 2008-10-05 06:00:00 2008-10-05 12:00:00  0.087917   
9  2008-10-05 06:00:00 2008-10-05 12:00:00 2008-10-05 18:00:00  0.086990   
10 2008-10-05 12:00:00 2008-10-05 18:00:00 2008-10-06 00:00:00  0.100428   
11 2008-10-05 18:00:0

In [7]:
# Cell 7: Step 9 (Acceptance Gates & Scientific Integrity) & Step 10 (Artifact Preservation)
gate_A_cov = (summary_A["coverage"] >= 0.70) if pd.notna(summary_A["coverage"]) else False
gate_A_wl = (summary_A["workload"] <= 0.30) if pd.notna(summary_A["workload"]) else False
gate_A_met = bool(gate_A_cov and gate_A_wl)

gate_B_rec = (metrics_B["recall"] >= 0.70) if pd.notna(metrics_B["recall"]) else False
gate_B_wl = (metrics_B["workload"] <= 0.30) if pd.notna(metrics_B["workload"]) else False
gate_B_met = bool(gate_B_rec and gate_B_wl)

production_ready = False

acceptance_report = {
    "branch_A_forecast": {
        "coverage": float(summary_A["coverage"]),
        "coverage_target_met": gate_A_cov,
        "workload": float(summary_A["workload"]),
        "workload_target_met": gate_A_wl,
        "overall_gate_passed": gate_A_met
    },
    "branch_B_qc_triage": {
        "raw_recall": float(metrics_B["recall"]),
        "recall_target_met": gate_B_rec,
        "workload": float(metrics_B["workload"]),
        "workload_target_met": gate_B_wl,
        "overall_gate_passed": gate_B_met
    },
    "production_ready": False,
    "operational_limitations": [
        "Retrospective evaluation: holdout observations were exposed in prior workspace exploratory phase.",
        "Low statistical sample size (104 total Fails in entire dataset; 16 in Test A, 17 in Holdout 314).",
        "Sensor feature and QC measurement delay (D=6h) are assumed engineering protocols, not measured log timestamps.",
        "Uncalibrated risk scores and event rankings do not represent physical machine health percentages or survival probabilities.",
        "No maintenance history, tool logs, or direct downtime data are present; operational savings cannot be asserted without prospective live-pilot validation."
    ]
}

print("=================================================================")
print("             ACCEPTANCE GATES EVALUATION REPORT                  ")
print("=================================================================")
print(f"Branch A Coverage >= 70%: {gate_A_cov} ({summary_A['coverage']:.4f})")
print(f"Branch A Workload <= 30%: {gate_A_wl} ({summary_A['workload']:.4f})")
print(f"Branch A Gate Passed    : {gate_A_met}")
print("-----------------------------------------------------------------")
print(f"Branch B Recall   >= 70%: {gate_B_rec} ({metrics_B['recall']:.4f})")
print(f"Branch B Workload <= 30%: {gate_B_wl} ({metrics_B['workload']:.4f})")
print(f"Branch B Gate Passed    : {gate_B_met}")
print("-----------------------------------------------------------------")
print(f"PRODUCTION_READY        : {production_ready}")
print("=================================================================")

# Step 10: Preserve in Version 2/experiment.joblib under 'final_evaluation_v2'
causal_checks = {
    "hashes_verified": True,
    "all_maturity_causal": True,
    "independent_union_verified": True,
    "B_prefix_quota_verified": True,
    "raw_ids_unique_and_aligned": True,
    "denominators_verified_fixed": True
}
prefix_reproduction = {
    "branch_A_dev_reproduced": True,
    "branch_B_dev_reproduced": True
}

exp["final_evaluation_v2"] = {
    "dev_replay_audit": compact_dev_report,
    "branch_A_summary": summary_A,
    "branch_A_baselines": baselines_A,
    "branch_A_bootstrap": ci_A,
    "branch_A_daily": daily_A.to_dict(),
    "branch_A_decisions": decisions_A_table,
    "branch_A_matured_audit": matured_audit_A,
    "branch_B_metrics": metrics_B,
    "branch_B_summary": summary_B,
    "branch_B_bootstrap": ci_B,
    "branch_B_daily": daily_B.to_dict(),
    "branch_B_decisions": decisions_B_table,
    "branch_B_matured_audit": matured_audit_B,
    "delay_robustness_dev": delay_robustness,
    "acceptance_gates": acceptance_report,
    "code_source_hashes": code_source_hashes,
    "stream_configs": {
        "branch_A": cfg_A,
        "branch_B": cfg_B
    },
    "causal_checks": causal_checks,
    "prefix_reproduction": prefix_reproduction,
    "selected_last_states": {
        "branch_A": last_state_A,
        "branch_B": last_state_B
    },
    "selected_traces": {
        "branch_A": trace_A,
        "branch_B": trace_B
    },
    "holdout_accounting": {
        "holdout_314_n": len(raw_314),
        "holdout_314_fail": int(raw_314["fail"].sum()),
        "pre_span_15_n": len(raw_pre),
        "pre_span_15_fail": int(raw_pre["fail"].sum()),
        "test_A_297_n": len(raw_test_A),
        "test_A_297_fail": int(raw_test_A["fail"].sum()),
        "post_span_2_n": len(raw_post),
        "post_span_2_fail": int(raw_post["fail"].sum())
    }
}

joblib.dump(exp, art_path)
print(f"[STEP 10] Successfully updated {art_path} with 'final_evaluation_v2'.")


             ACCEPTANCE GATES EVALUATION REPORT                  
Branch A Coverage >= 70%: False (0.5625)
Branch A Workload <= 30%: False (0.4478)
Branch A Gate Passed    : False
-----------------------------------------------------------------
Branch B Recall   >= 70%: False (0.2941)
Branch B Workload <= 30%: True (0.2739)
Branch B Gate Passed    : False
-----------------------------------------------------------------
PRODUCTION_READY        : False
[STEP 10] Successfully updated D:\AI_Bussiness_Huflit\Industrial_AI_on_UCISECOM\Version 2\experiment.joblib with 'final_evaluation_v2'.
